# Step 2 — Do code-switched ASR errors damage summaries? Conditions → Qwen summaries → silver facts → judge → stats

Paper: *How much do code-switched ASR errors damage summaries of Hinglish lecture audio?*

Inputs (from Step 1, on Drive): `selected_windows.csv`, `asr_hyps_vad15.csv`.

Pipeline (every stage is cached to Drive and resumable; re-running never repeats finished work):
1. **Silver references** (Gemini, from oracle transcripts): one English summary + 8–15 atomic facts per window.
   Each fact carries an evidence excerpt copied from the transcript; the excerpt is located in the transcript and
   the fact is tagged **hi / en / mixed** *deterministically* from the script of that span. These are SILVER data.
2. **Script-shift lexicon** (Gemini): Devanagari spelling of every English word in the oracle transcripts.
3. **Transcript conditions** (deterministic, seeded): oracle; real ASR (whisper-small, Trelis fine-tune);
   script-shift (English words rewritten in Devanagari, no word changed); noise injection on Hindi-only and on
   English-only tokens at 10–50%, using error patterns harvested from the real ASR output of the *other* windows.
   All conditions are fed in the same normalised form (lowercase, no punctuation).
4. **Summaries** (Qwen2.5-7B-Instruct, 4-bit, greedy): English for every condition; Hinglish (Roman script) as a
   secondary output for oracle + the two real ASR conditions.
5. **Fact judging** (same Gemini model as step 1, never the summariser): which silver facts each summary conveys.
6. **Metrics**: ROUGE-1/2/L and BERTScore after script normalisation (English summaries only); fact recall overall
   and by fact language; Code-Switching Loss gap; paired tests vs oracle with bootstrap CIs; sweep slopes;
   sensitivity analysis without W07/W24.
7. **Manual spot-check sheets** (10 reference windows, 30 judge decisions) + judge self-agreement.

**Runtime:** T4 GPU. Add your Gemini key in Colab **Secrets** (key icon, left bar) as `GEMINI_API_KEY`, notebook
access ON. The key is never printed or logged.

Gemini calls in this notebook: ~25 references + a few lexicon calls + ~475 judgements. If the free-tier daily quota
runs out, the stage stops cleanly; run the notebook again after the quota resets (midnight Pacific) and it continues.

In [ ]:
# ===================== CONFIG (edit here only) =====================
SEED = 13
ASR_TAG = "vad15"                                  # which Step-1 ASR outputs to use
SENSITIVITY_EXCLUDE = ["W07", "W24"]               # windows with VAD/truncation deletions (Step 1 v2)
# Number of windows evaluated in Step 2: the first N window ids (fixed by Step 1's seeded selection, chosen before any
# summary was scored). None = all. Reduced from 25 to 16 because of the Gemini free-tier quota.
STEP2_N_WINDOWS = 16

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/hinglish_asr_sum"
LOCAL_ROOT = "/content/hinglish_asr_sum"

# --- Gemini: reference writer AND fact judge (one model for both, never the summariser) ---
GEMINI_MODEL = "gemini-3.8-flash"     # pick a model that has free-tier quota in AI Studio > Rate limits
GEMINI_THINKING_LEVEL = None          # None = model default; e.g. "low" if supported by the model
MIN_SECONDS_BETWEEN_CALLS = 6.5       # ~9 requests/min; raise if you see many 429s
MAX_RETRIES = 8                       # per call, exponential backoff on 429 / 5xx
MAX_INVALID_JSON_RETRIES = 2
GEMINI_TIMEOUT_SEC = 180              # hard limit per request; a timeout is retried like a 5xx

# --- Summariser ---
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
QWEN_BATCH = 4
QWEN_MAX_NEW_TOKENS = 320

# --- Conditions ---
NOISE_LANGS = ["hi", "en"]
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5]          # fraction of that language's tokens corrupted
NOISE_SEED = 0
NOISE_PROFILE_MODEL = "trelis-hinglish-preview"   # S/D/I mix taken from this model's real errors
HINGLISH_OUTPUT_CONDITIONS = ["oracle", "asr:whisper-small", "asr:trelis-hinglish-preview"]

# --- Fact tagging from the evidence span's tokens (share of Devanagari among Hindi+English tokens) ---
TAG_HI_MIN = 0.6
TAG_EN_MAX = 0.4
EVIDENCE_MIN_MATCH = 0.90             # fuzzy match ratio for an evidence excerpt to count as located

# --- Evaluation ---
N_BOOT = 2000
SPOTCHECK_N_REF = 10
SPOTCHECK_N_JUDGE = 30
STABILITY_N = 20                      # summaries re-judged a second time to measure judge self-agreement

In [ ]:
# ===================== INSTALL + IMPORTS =====================
import subprocess, sys, importlib
PINS = ["httpx", "jiwer==4.0.0", "indic_transliteration==2.3.82", "rouge-score==0.1.2",
        "bert-score==0.3.13", "bitsandbytes", "accelerate"]
# Keep Colab's own GPU build of PyTorch: pin whatever torch/torchvision/torchaudio are already installed, so pip can
# never replace them with a CPU-only build while installing bitsandbytes / bert-score / accelerate.
import importlib.metadata as _md
KEEP_TORCH = []
for _p in ("torch", "torchvision", "torchaudio"):
    try:
        KEEP_TORCH.append(f"{_p}=={_md.version(_p)}")
    except _md.PackageNotFoundError:
        pass
print("keeping:", KEEP_TORCH)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS, *KEEP_TORCH], check=True)
importlib.invalidate_caches()

import os, re, gc, json, glob, time, math, random, shutil, hashlib, platform, unicodedata, difflib
from datetime import datetime, timezone
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from pydantic import BaseModel

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "NONE — Qwen and BERTScore need the T4")
try:
    import torch
    print(f"torch {torch.__version__} | built for CUDA {torch.version.cuda} | GPU usable: {torch.cuda.is_available()}")
    if gpu and not torch.cuda.is_available():
        print("WARNING: a GPU is attached but this torch cannot use it. Runtime > Disconnect and delete runtime, "
              "then Run all again.")
except ImportError:
    print("torch not installed")

In [ ]:
# ===================== PATHS, LOGGING, CACHE, STEP-1 INPUTS =====================
IN_COLAB = "google.colab" in sys.modules
ROOT = LOCAL_ROOT
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_ROOT
OUT_DIR, LOG_DIR = f"{ROOT}/outputs", f"{ROOT}/logs"
S2_DIR, CACHE_DIR, FIG_DIR = f"{ROOT}/outputs/step2", f"{ROOT}/cache", f"{ROOT}/outputs/step2/figures"
for d in (S2_DIR, f"{CACHE_DIR}/gemini", f"{CACHE_DIR}/qwen", FIG_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

PROMPT_LOG = f"{LOG_DIR}/prompts_log.jsonl"
RUN_CONFIG = f"{LOG_DIR}/run_config_step2.json"

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def log_prompt(step, component, model, prompt, params):
    rec = dict(ts=now_utc(), step=step, component=component, model=model, prompt=prompt, params=params)
    with open(PROMPT_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def update_run_config(**kv):
    cfg = json.load(open(RUN_CONFIG)) if os.path.exists(RUN_CONFIG) else {}
    cfg.update(kv)
    json.dump(cfg, open(RUN_CONFIG, "w"), indent=2, ensure_ascii=False, default=str)

def pkg_version(name):
    try:
        import importlib.metadata as md; return md.version(name)
    except Exception:
        return None

def sha(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

sel = pd.read_csv(f"{OUT_DIR}/selected_windows.csv", keep_default_na=False)
hyps = pd.read_csv(f"{OUT_DIR}/asr_hyps_{ASR_TAG}.csv", keep_default_na=False)
hyps = hyps.drop_duplicates(["window_id", "model"], keep="last")
WINDOWS = sorted(sel.window_id)
ASR_MODELS_FOUND = sorted(hyps.model.unique())
cnt = hyps.groupby("model").window_id.nunique()
assert (cnt == len(WINDOWS)).all(), f"incomplete ASR outputs: {cnt.to_dict()}"
sel_all, hyps_all, ALL_WINDOWS = sel.copy(), hyps.copy(), list(WINDOWS)   # all windows: used for noise patterns
if STEP2_N_WINDOWS:
    WINDOWS = ALL_WINDOWS[:STEP2_N_WINDOWS]
    sel = sel[sel.window_id.isin(WINDOWS)].reset_index(drop=True)
    hyps = hyps[hyps.window_id.isin(WINDOWS)].reset_index(drop=True)
print(f"Step 2 windows: {len(WINDOWS)} ({WINDOWS[0]}–{WINDOWS[-1]}) of {len(ALL_WINDOWS)} | ASR models: {ASR_MODELS_FOUND}")

update_run_config(started=now_utc(), python=platform.python_version(), gpu=gpu, seed=SEED,
                  config={k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, type(None)))},
                  packages={p: pkg_version(p) for p in ["google-genai", "transformers", "bitsandbytes", "accelerate",
                                                        "torch", "jiwer", "rouge-score", "bert-score", "pydantic"]})

In [ ]:
# ===================== TEXT UTILITIES (identical to Step 1) =====================
import jiwer
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

DEVA_RE = re.compile(r"[ऀ-ॿ]")
LATIN_RE = re.compile(r"[a-z]")
DEVA_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
DEVA_CONSONANTS = set(chr(c) for c in list(range(0x0915, 0x093A)) + list(range(0x0958, 0x0960)))
NUKTA = "़"

def norm_text(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"<[^>]*>|\[[^\]]*\]", " ", s).lower().translate(DEVA_DIGITS)
    s = "".join(ch if unicodedata.category(ch)[0] in "LMN" else " " for ch in s)
    return " ".join(s.split())

def token_script(tok):
    if DEVA_RE.search(tok):
        return "hi"
    if LATIN_RE.search(tok):
        return "en"
    return "other"

def romanize_token(tok):
    if not DEVA_RE.search(tok):
        return tok
    t = tok.replace(NUKTA, "")
    r = transliterate(t, sanscript.DEVANAGARI, sanscript.IAST)
    if len(t) > 1 and t[-1] in DEVA_CONSONANTS and r.endswith("a"):
        r = r[:-1]
    for a, b in (("ṃ", "n"), ("ṁ", "n"), ("~", "n"), ("ś", "sh"), ("ṣ", "sh"), ("ṛ", "ri"), ("ḥ", "h")):
        r = r.replace(a, b)
    r = re.sub(r"c(?!h)", "ch", r)
    r = "".join(ch for ch in unicodedata.normalize("NFD", r) if not unicodedata.combining(ch)).lower()
    r = re.sub(r"[^a-z0-9]", "", r)
    return r or tok

def romanize_text(normed):
    return " ".join(romanize_token(t) for t in normed.split())

def align_stats(ref, hyp):
    """WER and per-language error rates of hyp against ref (both normalised)."""
    rt = ref.split(); scripts = [token_script(t) for t in rt]
    tot = Counter(scripts)
    if not hyp.strip():
        err, S, D, I = Counter(scripts), 0, len(rt), 0
    else:
        o = jiwer.process_words(ref, hyp)
        err = Counter()
        for ch in o.alignments[0]:
            if ch.type != "equal":
                for i in range(ch.ref_start_idx, ch.ref_end_idx):
                    err[scripts[i]] += 1
        S, D, I = o.substitutions, o.deletions, o.insertions
    return dict(wer=(S + D + I) / len(rt), hi_err=err["hi"] / tot["hi"] if tot["hi"] else np.nan,
                en_err=err["en"] / tot["en"] if tot["en"] else np.nan, S=S, D=D, I=I)

assert romanize_text("क्या है निष्पादित statement") == "kya hai nishpadit statement"
sel["ref_norm"] = sel.ref_norm.map(norm_text)          # idempotent
sel_all["ref_norm"] = sel_all.ref_norm.map(norm_text)

In [ ]:
# ===================== GEMINI CLIENT: cache, pacing, retries, prompt log =====================
import httpx

# Direct REST call to the Interactions endpoint (https://ai.google.dev/gemini-api/docs/api-key shows this request).
# Reasons for not using the google-genai SDK here: AI Studio now issues "AQ." authentication keys, which some SDK
# paths reject (401 ACCESS_TOKEN_TYPE_UNSUPPORTED); and a plain HTTP call shows the server's exact error text.
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"
MAX_CONTINUATION_HOPS = 8

def _get_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("GEMINI_API_KEY")
            if k:
                return k.strip().strip('"').strip("'")
        except Exception:
            pass
    k = os.environ.get("GEMINI_API_KEY")
    return k.strip().strip('"').strip("'") if k else k

class GeminiHTTPError(Exception):
    def __init__(self, status_code, body):
        self.status_code = status_code
        super().__init__(f"HTTP {status_code}: {str(body)[:1500]}")

_http = httpx.Client(timeout=GEMINI_TIMEOUT_SEC)

def _post(body):
    key = _get_key()
    assert key, "Set GEMINI_API_KEY in Colab Secrets (or the environment)."
    r = _http.post(GEMINI_URL, headers={"x-goog-api-key": key, "Content-Type": "application/json"}, json=body)
    if r.status_code != 200:
        raise GeminiHTTPError(r.status_code, r.text)
    return r.json()

def _steps_text(steps):
    """Text of the final model output (same rule as the SDK's output_text: skip thoughts, stop at user input)."""
    parts, collecting = [], False
    for step in reversed(steps):
        t = step.get("type")
        if t == "user_input":
            break
        if t == "thought":
            continue
        if t != "model_output":
            if collecting:
                break
            continue
        stop = False
        for item in reversed(step.get("content") or []):
            if item.get("type") != "text":
                if collecting:
                    stop = True; break
                continue
            collecting = True
            parts.append(item.get("text") or "")
        if stop:
            break
    return "".join(reversed(parts))

def _api_create(req):
    """The only network call to Gemini. store=False (nothing kept server-side); if the server stops at its
    per-request decode limit (status requires_continuation), resend with the continuation token."""
    from types import SimpleNamespace
    body = {**req, "store": False}
    steps, d = [], {}
    for hop in range(1, MAX_CONTINUATION_HOPS + 1):
        d = _post(body)
        hs = d.get("steps") or []
        if hop > 1:
            while hs and hs[0].get("type") == "user_input":
                hs = hs[1:]
        steps += hs
        if d.get("status") == "requires_continuation" and d.get("continuation_token"):
            body = {**req, "store": False, "continuation_token": d["continuation_token"]}
            continue
        break
    if d.get("status") == "failed":
        raise GeminiHTTPError(503, f"interaction failed: {json.dumps(d.get('errors'))[:800]}")
    return SimpleNamespace(output_text=_steps_text(steps), model=d.get("model"), id=d.get("id"),
                           usage=d.get("usage"), status=d.get("status"), hops=hop)

def _response_format(schema):
    return {"type": "text", "mime_type": "application/json", "schema": schema.model_json_schema()}

def _model_version(m):
    g = re.search(r"gemini-(\d+(?:\.\d+)?)", m)
    return float(g.group(1)) if g else None

# Gemini 3.6+ and 4 ignore temperature (per the API docs, Oct 2026); send temperature=0 only to older models.
_v = _model_version(GEMINI_MODEL)
SEND_TEMPERATURE = _v is not None and _v < 3.6
print(f"Gemini model: {GEMINI_MODEL} | temperature=0 sent: {SEND_TEMPERATURE}"
      + ("" if SEND_TEMPERATURE else " (model ignores temperature; reproducibility comes from the response cache)"))

class GeminiStop(Exception):
    """Raised when retries are exhausted (usually the daily free-tier quota)."""

class GeminiAuthError(Exception):
    """Raised on 401/403: the key is missing, wrong or not a Gemini API key. Not retried."""

def check_gemini_key():
    """Shape check without revealing the key; then one tiny call so a bad key fails here, not mid-stage."""
    k = _get_key()
    assert k, "No GEMINI_API_KEY in Colab Secrets / environment."
    note = " | PROBLEM: looks like an OAuth access token, not an API key" if k.startswith("ya29.") else ""
    print(f"key: {len(k)} chars, starts '{k[:3]}…'{note}")
    class Ping(BaseModel):
        ok: bool
    try:
        r = _api_create(dict(model=GEMINI_MODEL, input='Return {"ok": true}.',
                             generation_config={"max_output_tokens": 1024},
                             response_format=_response_format(Ping), response_mime_type="application/json"))
        Ping.model_validate_json(r.output_text)
        print(f"Gemini key works with {GEMINI_MODEL} (response model: {r.model}, status: {r.status})")
    except Exception as e:
        code = getattr(e, "status_code", None)
        hint = {401: "key rejected: create a new key at https://aistudio.google.com/apikey and update the Secret",
                403: "key not allowed to use this API/model: check the key's project in AI Studio",
                404: "model not available to this key: change GEMINI_MODEL",
                400: "request rejected: send me this message",
                429: "rate limit / quota: wait and re-run"}.get(code, "send me this message")
        raise GeminiAuthError(f"Gemini test call failed ({type(e).__name__}): {str(e)[:1200]}\n=> {hint}") from None

_last_call = [0.0]
def gemini_json(template_id, system, user, schema, max_output_tokens, variant=0, meta=None):
    """Structured JSON call with disk cache. Returns (parsed pydantic object, cache record)."""
    gc_ = {"max_output_tokens": max_output_tokens}
    if SEND_TEMPERATURE:
        gc_["temperature"] = 0.0
    if GEMINI_THINKING_LEVEL:
        gc_["thinking_level"] = GEMINI_THINKING_LEVEL
    req = dict(model=GEMINI_MODEL, system_instruction=system, input=user, generation_config=gc_,
               response_format=_response_format(schema), response_mime_type="application/json")
    key = sha(dict(req=req, variant=variant))
    path = f"{CACHE_DIR}/gemini/{key}.json"
    # Reuse a response saved by the earlier notebook version (same model, prompts and schema; its request lacked
    # the two format fields added later, so its cache key differs). Copied under the new key, marked legacy.
    legacy_req = {k: v for k, v in req.items() if k != "response_mime_type"}
    legacy_req["response_format"] = {k: v for k, v in req["response_format"].items() if k != "type"}
    legacy_path = f"{CACHE_DIR}/gemini/{sha(dict(req=legacy_req, variant=variant))}.json"
    if not os.path.exists(path) and os.path.exists(legacy_path):
        try:
            old = json.load(open(legacy_path, encoding="utf-8"))
            schema.model_validate_json(old["output_text"])
            old.update(legacy_request_format=True, legacy_cache_file=os.path.basename(legacy_path))
            json.dump(old, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            print(f"   reused response from the earlier notebook version ({template_id}, {meta})", flush=True)
        except Exception as e:
            print(f"   could not reuse earlier response ({type(e).__name__}); calling Gemini", flush=True)
    if os.path.exists(path):
        rec = json.load(open(path, encoding="utf-8"))
        log_prompt("2", template_id, GEMINI_MODEL, None,
                   dict(cache_key=key, cache_hit=True, variant=variant, meta=meta))
        return schema.model_validate_json(rec["output_text"]), rec
    bad_json, attempt = 0, 0
    while True:
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        t_call = time.time()
        if template_id != "judge":
            print(f"   calling Gemini ({template_id}, {meta}) ...", flush=True)
        try:
            r = _api_create(req)
        except Exception as e:
            # The Interactions API raises its own classes (status in .status_code); older paths use .code.
            code = getattr(e, "status_code", None) or getattr(e, "code", None)
            transient = code in (429, 500, 502, 503, 504) or (code is None and type(e).__name__ in (
                "APIConnectionError", "APITimeoutError", "ConnectError", "ReadTimeout", "RemoteProtocolError",
                "TimeoutException", "WriteTimeout", "PoolTimeout", "ConnectTimeout"))
            if code in (401, 403):
                raise GeminiAuthError(
                    f"Gemini rejected the API key ({code}). Create a key at https://aistudio.google.com/apikey, "
                    "store it in Colab Secrets as GEMINI_API_KEY with notebook access ON, then re-run.") from e
            if transient and attempt < MAX_RETRIES:
                attempt += 1
                back = min(300, 10 * 2 ** (attempt - 1)) + random.Random(key + str(attempt)).uniform(0, 3)
                print(f"   Gemini {code or type(e).__name__}; retry {attempt}/{MAX_RETRIES} in {back:.0f}s")
                time.sleep(back)
                continue
            if transient:
                raise GeminiStop(f"Gemini {code or type(e).__name__} after {MAX_RETRIES} retries (usually the "
                                 "daily quota) — run again later; finished calls are cached.") from e
            raise
        text = getattr(r, "output_text", None) or ""
        try:
            parsed = schema.model_validate_json(text)
        except Exception:
            bad_json += 1
            if bad_json > MAX_INVALID_JSON_RETRIES:
                raise
            print(f"   invalid JSON from Gemini ({template_id}); retry {bad_json}")
            continue
        if template_id != "judge":
            print(f"   ... done in {time.time() - t_call:.0f}s", flush=True)
        usage = getattr(r, "usage", None)
        rec = dict(created=now_utc(), template_id=template_id, variant=variant, request=req, output_text=text,
                   response_model=getattr(r, "model", None), response_id=getattr(r, "id", None),
                   status=getattr(r, "status", None), hops=getattr(r, "hops", None),
                   usage=usage.model_dump() if hasattr(usage, "model_dump") else usage, meta=meta)
        json.dump(rec, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        log_prompt("2", template_id, GEMINI_MODEL, dict(system=system, input=user),
                   dict(cache_key=key, cache_hit=False, variant=variant, generation_config=gc_,
                        response_model=rec["response_model"], meta=meta))
        return parsed, rec

check_gemini_key()   # fails fast with a clear message if the key or model is wrong

In [ ]:
# ===================== PROMPT TEMPLATES (logged once per run) =====================
class Fact(BaseModel):
    fact: str
    evidence: str

class Reference(BaseModel):
    summary: str
    facts: list[Fact]

class Pair(BaseModel):
    word: str
    devanagari: str

class Lexicon(BaseModel):
    items: list[Pair]

class Judgement(BaseModel):
    id: int
    supported: bool

class Judgements(BaseModel):
    judgements: list[Judgement]

SYSTEM_REF = ("You are a careful annotator building evaluation data for summarisation research. "
              "Follow the instructions exactly and output only JSON that matches the schema.")
USER_REF = """Below is the exact human transcript of a segment of a recorded software tutorial. The speaker code-mixes Hindi and English: Hindi is mostly written in Devanagari and English mostly in Latin script, but some English words are written in Devanagari.

Produce two things.

1. "summary": an English summary of the segment's content, 120-180 words, plain prose, covering the main steps, concepts and facts taught. Do not mention the speaker, the tutorial series, greetings, credits, funding or websites.

2. "facts": 8 to 15 atomic facts that a good summary of this segment should contain.
- Each fact is one short, self-contained English sentence stating a single piece of content (a step, setting, definition, command or result).
- Exclude greetings, credits, acknowledgements, funding and website information.
- Facts must not overlap.
- For each fact, "evidence" must be copied EXACTLY, character for character, from the transcript: one contiguous excerpt of 3 to 25 words that states the fact. Do not translate, transliterate, correct or shorten words inside the excerpt.

Transcript:
<<<
{transcript}
>>>"""

SYSTEM_LEX = "You are a precise Hindi-English transliteration assistant. Output only JSON that matches the schema."
USER_LEX = """For each English word below, give the spelling a Hindi speaker would normally use when writing that English word in Devanagari script. Transliterate the pronunciation; do not translate the meaning. Examples: click -> क्लिक, file -> फ़ाइल, terminal -> टर्मिनल. For abbreviations, spell out the letter names (pdf -> पीडीएफ). Return exactly one item per input word, with the input word copied unchanged into "word".

Words:
{words}"""

SYSTEM_JUDGE = "You are a strict, consistent evaluator of summary content. Output only JSON that matches the schema."
USER_JUDGE = """Decide, for each numbered fact, whether the SUMMARY conveys it.

A fact is supported if the summary states its core information or clearly implies it, even in different words, in another language (English, Hindi or Hinglish) or in another script. It is not supported if it is missing, only vaguely related, or contradicted. Judge each fact independently, using only the summary.

SUMMARY:
<<<
{summary}
>>>

FACTS:
{facts}

Return exactly one judgement per fact id."""

SYSTEM_SUM = "You are a helpful assistant that summarises lecture transcripts."
USER_SUM_EN = """The following is an automatic transcript of part of a recorded computer-software tutorial. The speaker mixes Hindi and English (Hinglish); Hindi may be written in Devanagari and English in Latin script, and words of either language may appear in either script.

Write a summary in English of 120-180 words covering the main steps, concepts and facts taught. Write plain prose: no bullet points, no title.

Transcript:
<<<
{transcript}
>>>"""
USER_SUM_HING = USER_SUM_EN.replace(
    "Write a summary in English of 120-180 words",
    "Write a summary in Hinglish (Hindi-English mix written in Roman script, the way students commonly write) of 120-180 words")
assert USER_SUM_HING != USER_SUM_EN

TEMPLATES = dict(reference=(SYSTEM_REF, USER_REF), lexicon=(SYSTEM_LEX, USER_LEX), judge=(SYSTEM_JUDGE, USER_JUDGE),
                 summary_en=(SYSTEM_SUM, USER_SUM_EN), summary_hinglish=(SYSTEM_SUM, USER_SUM_HING))
for tid, (s, u) in TEMPLATES.items():
    log_prompt("2-templates", tid, GEMINI_MODEL if tid in ("reference", "lexicon", "judge") else QWEN_MODEL,
               dict(system=s, user_template=u), dict(template_sha=sha([s, u])))
update_run_config(templates={k: sha(v) for k, v in TEMPLATES.items()}, gemini_send_temperature=SEND_TEMPERATURE)

In [ ]:
# ===================== STAGE 1: SILVER REFERENCES + FACTS (Gemini) =====================
def locate_evidence(evidence, ref_tokens):
    """Find the evidence excerpt in the normalised transcript. Returns (start, end, ratio)."""
    ev = norm_text(evidence).split()
    if not ev:
        return None, None, 0.0
    n, L = len(ref_tokens), len(ev)
    joined_ev = " ".join(ev)
    for i in range(n - L + 1):                                 # exact token match first
        if ref_tokens[i:i + L] == ev:
            return i, i + L, 1.0
    best = (None, None, 0.0)
    for size in {max(1, L - 2), L - 1, L, L + 1, L + 2}:       # fuzzy: windows of similar length
        if size < 1:
            continue
        for i in range(0, max(0, n - size) + 1):
            r = difflib.SequenceMatcher(None, joined_ev, " ".join(ref_tokens[i:i + size]), autojunk=False).ratio()
            if r > best[2]:
                best = (i, i + size, r)
    return best

def tag_span(tokens):
    c = Counter(token_script(t) for t in tokens)
    lang = c["hi"] + c["en"]
    frac_hi = c["hi"] / lang if lang else np.nan
    tag = "hi" if frac_hi >= TAG_HI_MIN else ("en" if frac_hi <= TAG_EN_MAX else "mixed")
    return frac_hi, (tag if lang else "none")

ref_rows, fact_rows = [], []
stopped = None
print(f"cached Gemini responses on Drive: {len(glob.glob(f'{CACHE_DIR}/gemini/*.json'))}", flush=True)
for w in sel.itertuples():
    try:
        ref, rec = gemini_json("reference", SYSTEM_REF, USER_REF.format(transcript=w.ref_raw), Reference,
                               max_output_tokens=8192, meta=dict(window_id=w.window_id))
    except GeminiStop as e:
        stopped = str(e); break
    ref_tokens = w.ref_norm.split()
    t_loc = time.time()
    ref_rows.append(dict(window_id=w.window_id, ref_summary=ref.summary, n_facts_raw=len(ref.facts),
                         summary_words=len(ref.summary.split()), gemini_response_model=rec.get("response_model")))
    for k, f in enumerate(ref.facts):
        s, e, ratio = locate_evidence(f.evidence, ref_tokens)
        located = ratio >= EVIDENCE_MIN_MATCH
        frac_hi, tag = tag_span(ref_tokens[s:e]) if located else (np.nan, "unlocated")
        # English-term fact: an English (Latin-script) word spoken in the evidence also appears in the fact text
        fact_words = set(re.findall(r"[a-z0-9]+", f.fact.lower()))
        en_terms = sorted({t for t in (ref_tokens[s:e] if located else []) if token_script(t) == "en" and t in fact_words})
        fact_rows.append(dict(window_id=w.window_id, fact_id=f"{w.window_id}_F{k:02d}", fact=f.fact,
                              evidence=f.evidence, match_ratio=round(ratio, 3), located=located,
                              span_start=s, span_end=e, span_frac_hi=frac_hi, tag=tag,
                              has_en_term=bool(en_terms), en_terms=" ".join(en_terms)))
    n_loc = sum(r["located"] for r in fact_rows if r["window_id"] == w.window_id)
    print(f"{w.window_id}: {len(ref.facts)} facts, {n_loc} located (matching {time.time() - t_loc:.1f}s)", flush=True)
refs_df, facts_df = pd.DataFrame(ref_rows), pd.DataFrame(fact_rows)
refs_df.to_csv(f"{S2_DIR}/silver_references.csv", index=False)
facts_df.to_csv(f"{S2_DIR}/silver_facts.csv", index=False)
if stopped:
    print("STOPPED:", stopped)
print(f"references: {len(refs_df)}/{len(WINDOWS)} | facts: {len(facts_df)} | located: {facts_df.located.mean():.1%}")
print("fact tags:", facts_df.tag.value_counts().to_dict(),
      "| English-term facts:", int(facts_df[facts_df.located].has_en_term.sum()), "of", int(facts_df.located.sum()))
print("facts per window (located):", facts_df[facts_df.located].groupby("window_id").size().describe().round(1).to_dict())

In [ ]:
# ===================== STAGE 2: SCRIPT-SHIFT LEXICON (Gemini) =====================
en_types = sorted({t for r in sel.ref_norm for t in r.split() if token_script(t) == "en"})
LEX_BATCH = 150
lexicon, stopped = {}, None
for b in range(0, len(en_types), LEX_BATCH):
    words = en_types[b:b + LEX_BATCH]
    try:
        lx, _ = gemini_json("lexicon", SYSTEM_LEX, USER_LEX.format(words="\n".join(words)), Lexicon,
                            max_output_tokens=8192, meta=dict(batch=b // LEX_BATCH))
    except GeminiStop as e:
        stopped = str(e); break
    asked = set(words)
    for p in lx.items:
        w_ = norm_text(p.word); d_ = norm_text(p.devanagari)
        if w_ in asked and d_ and DEVA_RE.search(d_) and not LATIN_RE.search(d_) and len(d_.split()) <= 3:
            lexicon[w_] = d_.replace(" ", "")                    # keep token count identical
lex_df = pd.DataFrame(sorted(lexicon.items()), columns=["word", "devanagari"])
lex_df.to_csv(f"{S2_DIR}/script_shift_lexicon.csv", index=False)
if stopped:
    print("STOPPED:", stopped)
print(f"English word types: {len(en_types)} | lexicon entries: {len(lexicon)} ({len(lexicon) / max(1, len(en_types)):.1%})")
print("sample:", dict(list(lexicon.items())[:12]))

In [ ]:
# ===================== STAGE 3: TRANSCRIPT CONDITIONS (deterministic) =====================
# Error patterns are harvested from ALL Step-1 windows (more examples); conditions are built for WINDOWS only.
ref_by_w = sel_all.set_index("window_id").ref_norm.to_dict()
hyp_norm = {(h.window_id, h.model): norm_text(h.hyp_raw) for h in hyps_all.itertuples()}

# 3a. Error patterns from the real ASR output, per window (for leave-one-window-out sampling)
conf_by_w = defaultdict(lambda: defaultdict(Counter))    # window -> ref token -> Counter(hyp token)
ins_by_w = defaultdict(lambda: {"hi": Counter(), "en": Counter()})
sd_counts = {"hi": Counter(), "en": Counter()}             # S/D counts per language for the profile model
ins_total, err_total = 0, 0
for (wid, m), hyp in hyp_norm.items():
    rt, ht = ref_by_w[wid].split(), hyp.split()
    if not ht:
        continue
    o = jiwer.process_words(ref_by_w[wid], hyp)
    for ch in o.alignments[0]:
        if ch.type == "substitute":
            for k in range(ch.ref_end_idx - ch.ref_start_idx):
                r_, h_ = rt[ch.ref_start_idx + k], ht[ch.hyp_start_idx + k]
                conf_by_w[wid][r_][h_] += 1
                if m == NOISE_PROFILE_MODEL and token_script(r_) in sd_counts:
                    sd_counts[token_script(r_)]["S"] += 1
        elif ch.type == "delete" and m == NOISE_PROFILE_MODEL:
            for i in range(ch.ref_start_idx, ch.ref_end_idx):
                if token_script(rt[i]) in sd_counts:
                    sd_counts[token_script(rt[i])]["D"] += 1
        elif ch.type == "insert":
            for j in range(ch.hyp_start_idx, ch.hyp_end_idx):
                sc = token_script(ht[j])
                if sc in ("hi", "en"):
                    ins_by_w[wid][sc][ht[j]] += 1
    if m == NOISE_PROFILE_MODEL:
        ins_total += o.insertions; err_total += o.substitutions + o.deletions + o.insertions
P_INS = ins_total / err_total if err_total else 0.1
P_SUB = {l: (1 - P_INS) * sd_counts[l]["S"] / max(1, sd_counts[l]["S"] + sd_counts[l]["D"]) for l in ("hi", "en")}
NOISE_PROFILE = {l: dict(S=round(P_SUB[l], 4), D=round(1 - P_INS - P_SUB[l], 4), I=round(P_INS, 4)) for l in ("hi", "en")}
print("noise error-type mix (from", NOISE_PROFILE_MODEL, "):", NOISE_PROFILE)
VOCAB = {l: sorted({t for r in ref_by_w.values() for t in r.split() if token_script(t) == l}) for l in ("hi", "en")}

def char_corrupt(tok, rng):
    """Fallback substitution when a word was never misrecognised elsewhere: a 1-character edit."""
    pool = "कखगचजटडतदनपबमयरलवसह" if token_script(tok) == "hi" else "abcdefghiklmnoprstuvy"
    chars = list(tok)
    for _ in range(10):
        c = chars[:]
        i = rng.randrange(len(c)); op = rng.choice(["del", "rep", "ins"])
        if op == "del" and len(c) > 1:
            del c[i]
        elif op == "rep":
            c[i] = rng.choice(pool)
        else:
            c.insert(i, rng.choice(pool))
        out = unicodedata.normalize("NFC", "".join(c))
        if out != tok and out:
            return out
    return tok + pool[0]

def sample_counter(cnt, rng):
    items = sorted(cnt.items())
    total = sum(n for _, n in items)
    x = rng.uniform(0, total); acc = 0
    for k, n in items:
        acc += n
        if x <= acc:
            return k
    return items[-1][0]

def noisy_transcript(wid, lang, level):
    tokens = ref_by_w[wid].split()
    rng = random.Random(f"{SEED}|{wid}|{lang}|{NOISE_SEED}")
    pos = [i for i, t in enumerate(tokens) if token_script(t) == lang]
    rng.shuffle(pos)                                         # nested: level 0.2 corrupts a superset of level 0.1
    conf = defaultdict(Counter); ins_pool = Counter()        # leave-one-window-out error patterns
    for w2 in ALL_WINDOWS:
        if w2 != wid:
            for r_, c in conf_by_w[w2].items():
                conf[r_].update({h: n for h, n in c.items()})
            ins_pool.update(ins_by_w[w2][lang])
    plan = {}
    for i in pos:                                            # decide every position's corruption once
        x = rng.random(); p = NOISE_PROFILE[lang]
        if x < p["S"]:
            alt = Counter({h: n for h, n in conf[tokens[i]].items() if h != tokens[i]})
            plan[i] = ("S", sample_counter(alt, rng) if alt else char_corrupt(tokens[i], rng))
        elif x < p["S"] + p["D"]:
            plan[i] = ("D", None)
        else:
            plan[i] = ("I", sample_counter(ins_pool, rng) if ins_pool else rng.choice(VOCAB[lang]))
    chosen = set(pos[:round(level * len(pos))])
    n_lang = len(pos)
    out = []
    for i, t in enumerate(tokens):
        if i in chosen:
            op, val = plan[i]
            if op == "S":
                out.append(val)
            elif op == "I":
                out.extend([t, val])
        else:
            out.append(t)
    return " ".join(out), len(chosen), n_lang

cond_rows = []
for wid in WINDOWS:
    ref = ref_by_w[wid]
    def add(cond, family, text, lang=None, level=0.0, n_corrupted=0, n_lang=0):
        st = align_stats(ref, text)
        cond_rows.append(dict(window_id=wid, condition=cond, family=family, noise_lang=lang, level=level,
                              n_corrupted=n_corrupted, injected_rate=n_corrupted / n_lang if n_lang else 0.0,
                              text=text, n_words=len(text.split()),
                              cond_wer=st["wer"], cond_hi_err=st["hi_err"], cond_en_err=st["en_err"]))
    add("oracle", "oracle", ref)
    for m in ASR_MODELS_FOUND:
        add(f"asr:{m}", "asr", hyp_norm[(wid, m)])
    shifted = " ".join(lexicon.get(t, t) if token_script(t) == "en" else t for t in ref.split())
    add("script_shift", "script_shift", shifted)
    for lang in NOISE_LANGS:
        for lv in NOISE_LEVELS:
            txt, n, nl = noisy_transcript(wid, lang, lv)
            add(f"noise:{lang}:{lv:.1f}", "noise", txt, lang, lv, n, nl)
conds = pd.DataFrame(cond_rows)
conds.to_csv(f"{S2_DIR}/conditions.csv", index=False)
# injected_rate = corrupted tokens / tokens of that language (exact sweep variable). The measured per-language
# error rates are lower than injected_rate because an inserted word leaves the original token in place.
print(conds.groupby("condition")[["injected_rate", "cond_wer", "cond_hi_err", "cond_en_err", "n_words"]]
      .mean().round(3).to_string())
log_prompt("2-conditions", "noise_injection", None, None,
           dict(levels=NOISE_LEVELS, langs=NOISE_LANGS, seed=NOISE_SEED, profile=NOISE_PROFILE,
                profile_model=NOISE_PROFILE_MODEL, method="leave-one-window-out confusion sampling, nested levels"))

In [ ]:
# ===================== STAGE 4a: LOAD QWEN (GPU) =====================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import model_info

QWEN_REVISION = model_info(QWEN_MODEL).sha
qtok = AutoTokenizer.from_pretrained(QWEN_MODEL, revision=QWEN_REVISION, padding_side="left")
qmodel = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL, revision=QWEN_REVISION, device_map="cuda:0", torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
qmodel.eval()
QWEN_GEN = dict(max_new_tokens=QWEN_MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None)
update_run_config(qwen=dict(model=QWEN_MODEL, revision=QWEN_REVISION, quantization="bnb nf4 4-bit, fp16 compute",
                            generation=QWEN_GEN, batch=QWEN_BATCH))

def qwen_generate_batch(user_prompts):
    texts = [qtok.apply_chat_template([{"role": "system", "content": SYSTEM_SUM}, {"role": "user", "content": u}],
                                      tokenize=False, add_generation_prompt=True) for u in user_prompts]
    enc = qtok(texts, return_tensors="pt", padding=True).to(qmodel.device)
    torch.manual_seed(SEED)
    with torch.no_grad():
        out = qmodel.generate(**enc, **QWEN_GEN, pad_token_id=qtok.eos_token_id)
    return [qtok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out], \
           [int(m.sum()) for m in enc.attention_mask]

In [ ]:
# ===================== STAGE 4b: SUMMARISE ALL CONDITIONS (cached, prioritised) =====================
PRIORITY = {"oracle": 0, "asr": 1, "script_shift": 2, "noise": 3}
jobs = []
for c in conds.itertuples():
    jobs.append(dict(window_id=c.window_id, condition=c.condition, family=c.family, out_lang="en", text=c.text))
    if c.condition in HINGLISH_OUTPUT_CONDITIONS:
        jobs.append(dict(window_id=c.window_id, condition=c.condition, family=c.family, out_lang="hinglish", text=c.text))
for j in jobs:
    tmpl = USER_SUM_EN if j["out_lang"] == "en" else USER_SUM_HING
    j["user"] = tmpl.format(transcript=j["text"])
    j["key"] = sha(dict(model=QWEN_MODEL, rev=QWEN_REVISION, system=SYSTEM_SUM, user=j["user"], gen=QWEN_GEN))
    j["prio"] = (PRIORITY[j["family"]] + (5 if j["out_lang"] == "hinglish" else 0), j["condition"], j["window_id"])
jobs.sort(key=lambda j: j["prio"])
todo = [j for j in jobs if not os.path.exists(f"{CACHE_DIR}/qwen/{j['key']}.json")]
print(f"summaries: {len(jobs)} total, {len(todo)} to generate")
t_start = time.time()
for b in range(0, len(todo), QWEN_BATCH):
    batch = todo[b:b + QWEN_BATCH]
    batch.sort(key=lambda j: len(j["user"]))
    t0 = time.time()
    outs, n_in = qwen_generate_batch([j["user"] for j in batch])
    for j, o, n in zip(batch, outs, n_in):
        rec = dict(created=now_utc(), model=QWEN_MODEL, revision=QWEN_REVISION, gen=QWEN_GEN, batch=QWEN_BATCH,
                   window_id=j["window_id"], condition=j["condition"], out_lang=j["out_lang"],
                   input_tokens=n, summary=o)
        json.dump(rec, open(f"{CACHE_DIR}/qwen/{j['key']}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        log_prompt("2", f"summary_{j['out_lang']}", QWEN_MODEL, dict(system=SYSTEM_SUM, user=j["user"]),
                   dict(cache_key=j["key"], window_id=j["window_id"], condition=j["condition"], gen=QWEN_GEN))
    done_n = b + len(batch)
    rate = (time.time() - t_start) / done_n
    print(f"  {done_n}/{len(todo)} | {time.time() - t0:.0f}s this batch | ~{rate * (len(todo) - done_n) / 60:.0f} min left")

rows = []
for j in jobs:
    p = f"{CACHE_DIR}/qwen/{j['key']}.json"
    if os.path.exists(p):
        r = json.load(open(p, encoding="utf-8"))
        rows.append(dict(window_id=j["window_id"], condition=j["condition"], family=j["family"],
                         out_lang=j["out_lang"], summary=r["summary"], input_tokens=r["input_tokens"],
                         summary_words=len(r["summary"].split()), summary_key=j["key"]))
sums = pd.DataFrame(rows)
sums.to_csv(f"{S2_DIR}/summaries.csv", index=False)
print(sums.groupby(["out_lang", "family"]).summary_words.describe()[["count", "mean", "min", "max"]].round(0))

In [ ]:
# ===================== STAGE 4c: FREE GPU MEMORY =====================
try:
    del qmodel; gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass

In [ ]:
# ===================== STAGE 5: JUDGE FACTS (Gemini, cached, prioritised) =====================
facts_ok = facts_df[facts_df.located].copy()
facts_by_w = {w: g.reset_index(drop=True) for w, g in facts_ok.groupby("window_id")}

def judge(summary, wid, variant=0, meta=None):
    fl = facts_by_w[wid]
    facts_txt = "\n".join(f"{i + 1}. {f}" for i, f in enumerate(fl.fact))
    res, rec = gemini_json("judge", SYSTEM_JUDGE, USER_JUDGE.format(summary=summary, facts=facts_txt), Judgements,
                           max_output_tokens=4096, variant=variant, meta=meta)
    got = {j.id: j.supported for j in res.judgements}
    if sorted(got) != list(range(1, len(fl) + 1)):
        print(f"   WARNING {wid}: judge returned ids {sorted(got)} for {len(fl)} facts; missing ids -> NaN")
    return [dict(fact_id=fid, supported=got.get(i + 1)) for i, fid in enumerate(fl.fact_id)]

judge_items = [dict(kind="summary", window_id=s.window_id, condition=s.condition, out_lang=s.out_lang,
                    summary=s.summary, prio=(PRIORITY[s.family] + (5 if s.out_lang == "hinglish" else 0), s.condition))
               for s in sums.itertuples() if s.window_id in facts_by_w]
judge_items += [dict(kind="reference_sanity", window_id=r.window_id, condition="silver_reference", out_lang="en",
                     summary=r.ref_summary, prio=(9, "silver_reference")) for r in refs_df.itertuples()
                if r.window_id in facts_by_w]
judge_items.sort(key=lambda d: (d["prio"], d["window_id"]))
jrows, stopped = [], None
for k, it in enumerate(judge_items):
    try:
        res = judge(it["summary"], it["window_id"], meta=dict(window_id=it["window_id"], condition=it["condition"],
                                                               out_lang=it["out_lang"], kind=it["kind"]))
    except GeminiStop as e:
        stopped = str(e); break
    for r in res:
        jrows.append(dict(window_id=it["window_id"], condition=it["condition"], out_lang=it["out_lang"],
                          kind=it["kind"], **r))
    if k % 25 == 0:
        print(f"  judged {k + 1}/{len(judge_items)}")
judg = pd.DataFrame(jrows)
judg.to_csv(f"{S2_DIR}/judgements.csv", index=False)
if stopped:
    print("STOPPED:", stopped, f"({judg[['window_id', 'condition', 'out_lang']].drop_duplicates().shape[0]} summaries judged)")

# Judge self-agreement: re-judge a seeded sample a second time (separate cache entry)
stab_rows = []
if not stopped:
    rng_s = random.Random(SEED)
    pool = [it for it in judge_items if it["kind"] == "summary"]
    for it in rng_s.sample(pool, min(STABILITY_N, len(pool))):
        try:
            res = judge(it["summary"], it["window_id"], variant=1, meta=dict(stability=True, **{k: it[k] for k in
                        ("window_id", "condition", "out_lang")}))
        except GeminiStop as e:
            print("STOPPED during stability check:", e); break
        for r in res:
            stab_rows.append(dict(window_id=it["window_id"], condition=it["condition"], out_lang=it["out_lang"],
                                  fact_id=r["fact_id"], supported_2=r["supported"]))
if stab_rows:
    st = pd.DataFrame(stab_rows).merge(judg[judg.kind == "summary"], on=["window_id", "condition", "out_lang", "fact_id"])
    st = st.dropna(subset=["supported", "supported_2"])
    agree = (st.supported == st.supported_2).mean()
    st.to_csv(f"{S2_DIR}/judge_stability.csv", index=False)
    print(f"judge self-agreement on {len(st)} fact decisions ({STABILITY_N} summaries judged twice): {agree:.1%}")
    update_run_config(judge_self_agreement=dict(n=len(st), agreement=float(agree)))

In [ ]:
# ===================== STAGE 6a: BERTSCORE LOADER =====================
from bert_score import score as _bscore
BERTSCORE_MODEL = "roberta-large"
def bertscore_f1(cands, refs):
    _, _, F = _bscore(cands, refs, model_type=BERTSCORE_MODEL, lang="en", batch_size=16, verbose=False)
    return F.tolist()
update_run_config(bertscore=dict(model=BERTSCORE_MODEL, rescale_with_baseline=False))

In [ ]:
# ===================== STAGE 6b: PER-SUMMARY METRICS =====================
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
prep = lambda s: romanize_text(norm_text(s))          # script normalisation before ROUGE / BERTScore
ref_sum = refs_df.set_index("window_id").ref_summary.to_dict()

m = sums[sums.window_id.isin(ref_sum)].copy()
en_mask = m.out_lang == "en"
for k in ("rouge1", "rouge2", "rougeL", "bertscore_f1"):
    m[k] = np.nan
for i in m[en_mask].index:
    sc = scorer.score(prep(ref_sum[m.at[i, "window_id"]]), prep(m.at[i, "summary"]))
    for k in ("rouge1", "rouge2", "rougeL"):
        m.at[i, k] = sc[k].fmeasure
idx = list(m[en_mask].index)
if idx:
    m.loc[idx, "bertscore_f1"] = bertscore_f1([prep(m.at[i, "summary"]) for i in idx],
                                              [prep(ref_sum[m.at[i, "window_id"]]) for i in idx])

jj = judg[judg.kind == "summary"].merge(facts_ok[["fact_id", "tag", "has_en_term"]], on="fact_id") \
         .dropna(subset=["supported"])
jj["supported"] = jj.supported.astype(bool)
rec = jj.groupby(["window_id", "condition", "out_lang"]).supported.mean().rename("recall_all")
by_tag = jj.groupby(["window_id", "condition", "out_lang", "tag"]).supported.mean().unstack("tag")
by_tag.columns = [f"recall_{c}" for c in by_tag.columns]
n_tag = jj.groupby(["window_id", "condition", "out_lang", "tag"]).size().unstack("tag").fillna(0).astype(int)
n_tag.columns = [f"n_facts_{c}" for c in n_tag.columns]
m = m.merge(rec, on=["window_id", "condition", "out_lang"], how="left") \
     .merge(by_tag, on=["window_id", "condition", "out_lang"], how="left") \
     .merge(n_tag, on=["window_id", "condition", "out_lang"], how="left")
for c in ("recall_hi", "recall_en", "recall_mixed"):
    if c not in m:
        m[c] = np.nan
m["csl_gap"] = m.recall_en - m.recall_hi              # >0: English-spoken facts survive better than Hindi-spoken
et = jj.groupby(["window_id", "condition", "out_lang", "has_en_term"]).supported.mean().unstack("has_en_term")
et = et.rename(columns={True: "recall_enterm", False: "recall_noenterm"})
for c in ("recall_enterm", "recall_noenterm"):
    if c not in et:
        et[c] = np.nan
m = m.merge(et[["recall_enterm", "recall_noenterm"]], on=["window_id", "condition", "out_lang"], how="left")
m["enterm_gap"] = m.recall_enterm - m.recall_noenterm  # >0: facts carried by an English term survive better
m = m.merge(conds.drop(columns=["text"]), on=["window_id", "condition", "family"], how="left")
m["sensitivity_excluded"] = m.window_id.isin(SENSITIVITY_EXCLUDE)
m.to_csv(f"{S2_DIR}/summary_metrics_per_item.csv", index=False)

sanity = judg[judg.kind == "reference_sanity"].dropna(subset=["supported"])
if len(sanity):
    print(f"judge ceiling (silver summary vs its own facts): recall {sanity.supported.astype(bool).mean():.1%}")
print(m.groupby(["out_lang", "condition"])[["rougeL", "bertscore_f1", "recall_all", "recall_hi", "recall_en",
                                             "csl_gap"]].mean().round(3).to_string())

In [ ]:
# ===================== STAGE 7: STATISTICS =====================
from scipy.stats import wilcoxon
METRICS = ["rouge1", "rouge2", "rougeL", "bertscore_f1", "recall_all", "recall_hi", "recall_en", "csl_gap",
           "recall_enterm", "recall_noenterm", "enterm_gap"]

def holm(ps):
    ps = np.asarray(ps, float); out = np.full_like(ps, np.nan)
    ok = ~np.isnan(ps); idx = np.argsort(ps[ok]); vals = ps[ok][idx]; k = len(vals)
    adj = np.maximum.accumulate([(k - i) * v for i, v in enumerate(vals)]).clip(max=1)
    tmp = np.empty(k); tmp[idx] = adj; out[ok] = tmp
    return out

def paired_vs_oracle(df, subset_name):
    rows = []
    for ol in sorted(df.out_lang.unique()):
        base = df[(df.out_lang == ol) & (df.condition == "oracle")].set_index("window_id")
        for cond in sorted(set(df[df.out_lang == ol].condition) - {"oracle"}):
            cur = df[(df.out_lang == ol) & (df.condition == cond)].set_index("window_id")
            for met in METRICS:
                a = cur[met].reindex(base.index); b = base[met]
                ok = a.notna() & b.notna()
                if ok.sum() < 3:
                    continue
                d = (a[ok] - b[ok]).to_numpy()
                bs = np.random.default_rng(SEED).integers(0, len(d), (N_BOOT, len(d)))
                try:
                    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
                except ValueError:
                    p = np.nan
                rows.append(dict(subset=subset_name, out_lang=ol, condition=cond, metric=met, n_windows=int(ok.sum()),
                                 oracle_mean=b[ok].mean(), cond_mean=a[ok].mean(), diff=d.mean(),
                                 ci95_lo=np.percentile(d[bs].mean(1), 2.5), ci95_hi=np.percentile(d[bs].mean(1), 97.5),
                                 wilcoxon_p=p))
    out = pd.DataFrame(rows)
    if len(out):
        out["p_holm_within_metric"] = out.groupby(["out_lang", "metric"]).wilcoxon_p.transform(lambda s: holm(s.values))
    return out

paired = pd.concat([paired_vs_oracle(m, "all_windows"),
                    paired_vs_oracle(m[~m.sensitivity_excluded], "excl_" + "_".join(SENSITIVITY_EXCLUDE))])
paired.to_csv(f"{S2_DIR}/paired_vs_oracle.csv", index=False)

def sweep_slopes(df, subset_name):
    """OLS slope of each metric on the injected rate (and on measured overall WER); oracle = 0; bootstrap windows."""
    rows = []
    en = df[df.out_lang == "en"]
    for lang in NOISE_LANGS:
        part = en[(en.condition == "oracle") | ((en.family == "noise") & (en.noise_lang == lang))].copy()
        part["x"] = np.where(part.condition == "oracle", 0.0, part.injected_rate)
        part["x_wer"] = np.where(part.condition == "oracle", 0.0, part.cond_wer)
        wins = sorted(part.window_id.unique())
        g = {w: part[part.window_id == w] for w in wins}
        rng = np.random.default_rng(SEED)
        for met in METRICS:
            for xcol in ("x", "x_wer"):
                def slope(ws):
                    d = pd.concat([g[w] for w in ws]).dropna(subset=[met, xcol])
                    return np.polyfit(d[xcol], d[met], 1)[0] if d[xcol].nunique() > 1 else np.nan
                pt = slope(wins)
                bs = [slope([wins[i] for i in rng.integers(0, len(wins), len(wins))]) for _ in range(N_BOOT // 4)]
                rows.append(dict(subset=subset_name, sweep=lang, metric=met,
                                 x=("injected_" + lang + "_rate") if xcol == "x" else "measured_overall_wer",
                                 slope_per_unit=pt, ci95_lo=np.nanpercentile(bs, 2.5), ci95_hi=np.nanpercentile(bs, 97.5),
                                 n_windows=len(wins)))
    return pd.DataFrame(rows)

slopes = pd.concat([sweep_slopes(m, "all_windows"),
                    sweep_slopes(m[~m.sensitivity_excluded], "excl_" + "_".join(SENSITIVITY_EXCLUDE))])
slopes.to_csv(f"{S2_DIR}/sweep_slopes.csv", index=False)

show = paired[(paired.subset == "all_windows") & paired.metric.isin(["recall_all", "recall_hi", "recall_en", "rougeL"])
              & ~paired.condition.str.startswith("noise")]
print(show[["out_lang", "condition", "metric", "oracle_mean", "cond_mean", "diff", "ci95_lo", "ci95_hi",
            "wilcoxon_p", "p_holm_within_metric"]].round(4).to_string(index=False))
print(slopes[(slopes.subset == "all_windows") & slopes.metric.isin(["recall_all", "recall_hi", "recall_en"])]
      .round(4).to_string(index=False))

In [ ]:
# ===================== STAGE 8: FIGURES =====================
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

def mean_ci(v):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    bs = np.random.default_rng(SEED).choice(v, (N_BOOT, len(v))).mean(1)
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

en = m[m.out_lang == "en"]
fig, axes = plt.subplots(1, len(NOISE_LANGS), figsize=(9, 3.4), sharey=True)
for ax, lang in zip(np.atleast_1d(axes), NOISE_LANGS):
    for met, color, label in (("recall_hi", "#1f77b4", "Hindi-spoken facts"), ("recall_en", "#d62728", "English-spoken facts")):
        xs, ys, lo, hi = [], [], [], []
        for lv in [0.0] + NOISE_LEVELS:
            cond = "oracle" if lv == 0 else f"noise:{lang}:{lv:.1f}"
            mu, a, b = mean_ci(en[en.condition == cond][met])
            xs.append(lv * 100); ys.append(mu); lo.append(a); hi.append(b)
        ax.plot(xs, ys, marker="o", color=color, label=label)
        ax.fill_between(xs, lo, hi, color=color, alpha=0.15, linewidth=0)
    ax.set_title(f"Noise on {'Hindi' if lang == 'hi' else 'English'} tokens only")
    ax.set_xlabel(f"% of {'Hindi' if lang == 'hi' else 'English'} tokens corrupted")
np.atleast_1d(axes)[0].set_ylabel("Fact recall (silver facts)")
np.atleast_1d(axes)[0].legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_noise_sweeps.png", dpi=200); plt.show()

order = ["oracle", "script_shift"] + [f"asr:{x}" for x in ASR_MODELS_FOUND]
fig, ax = plt.subplots(figsize=(7, 3.4))
w_ = 0.38
for k, (met, color, label) in enumerate((("recall_hi", "#1f77b4", "Hindi-spoken facts"),
                                         ("recall_en", "#d62728", "English-spoken facts"))):
    stats_ = [mean_ci(en[en.condition == c][met]) for c in order]
    mu = [s[0] for s in stats_]
    err = [[s[0] - s[1] for s in stats_], [s[2] - s[0] for s in stats_]]
    ax.bar(np.arange(len(order)) + (k - 0.5) * w_, mu, w_, yerr=err, color=color, alpha=0.85, label=label, capsize=3)
ax.set_xticks(range(len(order)), [o.replace("asr:", "") for o in order], rotation=15)
ax.set_ylabel("Fact recall (silver facts)"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig_conditions.png", dpi=200); plt.show()

In [ ]:
# ===================== STAGE 9: MANUAL SPOT-CHECK SHEETS + AGREEMENT =====================
rng_sc = random.Random(SEED)
sc_w = sorted(rng_sc.sample(sorted(refs_df.window_id), min(SPOTCHECK_N_REF, len(refs_df))))
sc = facts_df[facts_df.window_id.isin(sc_w)].merge(refs_df[["window_id", "ref_summary"]], on="window_id")
sc = sc[["window_id", "ref_summary", "fact_id", "fact", "evidence", "located", "tag", "span_frac_hi"]].copy()
for c in ("fact_correct_y_n", "tag_correct_y_n", "summary_faithful_y_n", "notes"):
    sc[c] = ""
SC_REF = f"{S2_DIR}/spotcheck_references.csv"
if not os.path.exists(SC_REF):
    sc.to_csv(SC_REF, index=False)

jsum = judg[judg.kind == "summary"].dropna(subset=["supported"])
if len(jsum):
    js = jsum.sample(min(SPOTCHECK_N_JUDGE, len(jsum)), random_state=SEED) \
             .merge(sums[["window_id", "condition", "out_lang", "summary"]], on=["window_id", "condition", "out_lang"]) \
             .merge(facts_ok[["fact_id", "fact"]], on="fact_id")
    js = js[["window_id", "condition", "out_lang", "summary", "fact_id", "fact", "supported"]].rename(
        columns={"supported": "judge_supported"})
    js["human_supported_y_n"] = ""
    SC_J = f"{S2_DIR}/spotcheck_judge.csv"
    if not os.path.exists(SC_J):
        js.to_csv(SC_J, index=False)
print("Spot-check sheets:", SC_REF, f"{S2_DIR}/spotcheck_judge.csv")

def yn(s):
    s = str(s).strip().lower()
    return True if s in ("y", "yes", "1", "true") else (False if s in ("n", "no", "0", "false") else np.nan)

for fname in ("spotcheck_references_filled.csv", "spotcheck_judge_filled.csv"):
    p = f"{S2_DIR}/{fname}"
    if not os.path.exists(p):
        print(f"(fill in and save as {fname} to compute agreement)"); continue
    f = pd.read_csv(p, keep_default_na=False)
    if "human_supported_y_n" in f:
        h = f.human_supported_y_n.map(yn); j = f.judge_supported.astype(str).str.lower().eq("true")
        ok = h.notna(); a = (h[ok] == j[ok]).mean()
        po, pe = a, (h[ok].mean() * j[ok].mean() + (1 - h[ok].mean()) * (1 - j[ok].mean()))
        kappa = (po - pe) / (1 - pe) if pe < 1 else np.nan
        print(f"judge vs human: n={ok.sum()} agreement={a:.1%} Cohen's kappa={kappa:.2f}")
        update_run_config(judge_vs_human=dict(n=int(ok.sum()), agreement=float(a), kappa=float(kappa)))
    else:
        out = {c: f[c].map(yn).mean() for c in ("fact_correct_y_n", "tag_correct_y_n") if c in f}
        summ_ok = f.drop_duplicates("window_id").summary_faithful_y_n.map(yn).mean()
        print("reference spot-check:", {k: round(v, 3) for k, v in out.items()}, "| summaries faithful:", round(summ_ok, 3))
        update_run_config(reference_spotcheck=dict(**{k: float(v) for k, v in out.items()}, summary_faithful=float(summ_ok)))
update_run_config(finished=now_utc())

## Before writing up — check these

1. **Reference stage:** `located` should be high (> 85%). Unlocated facts are excluded from recall; if many are
   unlocated, read a few `evidence` strings — the writer may have translated instead of copying.
2. **Fact tags:** look at the `fact tags` counts. If `mixed` dominates, Hindi-vs-English comparisons rest on few
   facts; report the counts next to every CSL number.
3. **Lexicon:** read 20 rows of `script_shift_lexicon.csv`; wrong transliterations make the script-shift condition
   harder than real Devanagari-English, so note any systematic problem.
4. **Judge ceiling** (silver summary vs its own facts) should be near 100%; much lower means the judge is too strict.
5. **Judge self-agreement** (printed in Stage 5) is your noise floor: differences smaller than
   (1 − agreement) are not interpretable.
6. **Spot-checks:** fill `spotcheck_references.csv` and `spotcheck_judge.csv` (y/n), save as `*_filled.csv`, re-run
   Stage 9. Report the numbers in the paper. Call references "silver" everywhere.
7. **Sensitivity:** every claim should hold in both `all_windows` and `excl_W07_W24`; say so or report the
   disagreement.
8. **Logs:** `prompts_log.jsonl` (all templates + every call), `run_config_step2.json` (models, revisions,
   versions, judge temperature handling) and `cache/` (every raw response) go into the appendix / supplementary.